In [9]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import pytest

In [10]:
from qrc_bloqade.encodings import amp_encode, angle_encode
from qrc_bloqade.hamiltonian import rydberg 
from qrc_bloqade.readouts import ZReadout
from qrc_bloqade.solver import BloqadeSolver 
from qrc_bloqade.utils import(train_split, normalize, get_predictor, print_readout) 


In [11]:
@pytest.mark.parametrize("encoding_type", ['amp', 'angle'])
def test_bb(encoding_type):
    n_sites=4
    omega=1.0
    V=1.0
    time=1.0

    hamiltonian= rydberg(n_sites, omega, V)
    readout= ZReadout(n_sites)
    predictior= get_predictor()
    scaler = MinMaxScaler(feature_range=(-1, 1))
    splitter = train_split()

    X,y= make_regression(n_samples=1000, n_features=n_sites, noise=0.1, random_state= 403)

    if encoding_type=="amp":
        encoder= amp_encode(n_sites)
        X_normalized, y_normalized, scalar_X, scaler_y= normalize(X, y)

    elif encoding_type=="angle":
        encoder= angle_encode(n_sites)
         # Normalize data for Amplitude Encoding (sum of squares = 1)
        X_normalized=[]
        for row in X:
            norm = np.linalg.norm(row) #for 3,4 it gives 5
            X_normalized.append(row/norm) #adds 3/5,4/5:normalized form to new array
        X_normalized= np.array(X_normalized)

        #padding
        X_normalized= np.pad(X_normalized, ((0,0),(0, 2**n_sites- X_normalized.shape[1])), 'constant')
        
         #Normalize y and keep scaler
        y_normalized, scaler_y = normalize(X_normalized,y)[:2] #Normalize and get scaler

        # Check if data has a valid shape after padding
        if X_normalized.shape[1] != 2**n_sites:
            raise ValueError(
                f"Data has wrong shape after padding {X_normalized.shape[1]}. "
                f"Expected {2**n_sites}."
            )
    else:
        raise ValueError(f"Invalid encoding type: {encoding_type}")


    # Split data
    X_train, X_val, X_test, y_train, y_val, y_test = splitter(
        X_normalized, y_normalized, test_size=0.4, random_state=402
    )

    # --- QRC Workflow ---
    # Encode
    encoded_train = encoder.encode(X_train)
    encoded_test = encoder.encode(X_test)

    # Evolve (Dynamics)
    solver = BloqadeSolver()
    evolved_train = solver.simulate(
        encoded_train, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100
    )
    evolved_test = solver.simulate(
        encoded_test, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100
    )

    # Measure
    train_features = readout.measure(evolved_train)
    test_features = readout.measure(evolved_test)

    # Predict (Train and Test)
    predictor.fit(train_features, y_train)
    predictions = predictor.predict(test_features)

    # Inverse transform to get original scale

    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    # Evaluate
    mse = mean_squared_error(y_test, predictions)

    print(f"Test MSE: {mse}")
    assert mse >= 0  # Basic check: MSE should be non-negative

    # --- Manual Feature Selection Example (within the same test) ---
    selected_readout_indices = [0, 2]  # Measure qubits 0 and 2
    train_features_subset = readout.measure(
        evolved_train, readout_indices=selected_readout_indices
    )
    test_features_subset = readout.measure(
        evolved_test, readout_indices=selected_readout_indices
    )
    predictor = get_predictor()
    predictor.fit(
        train_features_subset, y_train
    )  # Fit on the subset of training features
    predictions_subset = predictor.predict(
        test_features_subset
    )  # Predict with test features
    predictions_subset = scaler_y.inverse_transform(predictions_subset.reshape(-1, 1)).flatten()

    mse_subset = mean_squared_error(y_test, predictions_subset)
    print(f"Test MSE (subset of readouts): {mse_subset}")
    assert mse_subset >= 0     


In [12]:
# Cell 1: Imports and Setup (All in one cell for clarity)

import numpy as np
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qrc_bloqade.encodings import amp_encode, angle_encode
from qrc_bloqade.hamiltonian import rydberg 
from qrc_bloqade.readouts import ZReadout
from qrc_bloqade.solver import BloqadeSolver 
from qrc_bloqade.utils import(train_split, normalize, get_predictor, print_readout)


# Cell 2: Test function

def test_qrc_angle_encoding():
    n_sites = 4
    omega = 1.0
    V = 1.0
    time = 1.0

    # Create instances of the concrete classes
    hamiltonian = rydberg(n_sites, omega, V)
    encoder = angle_encode(n_sites)  # Use AngleEncoding
    readout = ZReadout(n_sites)
    predictor = get_predictor()
    scaler = MinMaxScaler(feature_range=(-1, 1))
    splitter = train_split

    # Generate synthetic data
    X, y = make_regression(n_samples=100, n_features=n_sites, noise=0.1, random_state=402)

    # Normalize data
    X_normalized, y_normalized, scaler_X, scaler_y = normalize(X, y)

    # Split data
    X_train, X_val, X_test, y_train, y_val, y_test = splitter(
        X_normalized, y_normalized, test_size=0.4, random_state=402
    )

    # --- QRC Workflow ---
    # Encode
    encoded_train = encoder.encode(X_train)
    encoded_test = encoder.encode(X_test)

    # Evolve (Dynamics)
    solver = BloqadeSolver()
    evolved_train = solver.simulate(
        encoded_train, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100
    )
    evolved_test = solver.simulate(
        encoded_test, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100
    )

    # Measure
    train_features = readout.measure(evolved_train)
    test_features = readout.measure(evolved_test)

    # Predict (Train and Test)
    predictor.fit(train_features, y_train)
    predictions = predictor.predict(test_features)

    # Inverse transform to get original scale
    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    # Evaluate
    mse = mean_squared_error(y_test, predictions)

    print(f"Test MSE (Angle Encoding): {mse}")
    assert mse >= 0  # Basic check

#

In [13]:
# Cell 3: Test function (Amplitude Encoding)
def test_qrc_amplitude_encoding():
    n_sites = 2  # Use only 2 qubits for easier amplitude encoding
    omega = 1.0
    V = 1.0
    time = 1.0

    # QRC components
    hamiltonian = rydberg(n_sites, omega, V)
    encoder = amp_encode(n_sites)  # Use AmplitudeEncoding
    readout = ZReadout(n_sites)
    predictor = get_predictor()
    scaler = MinMaxScaler(feature_range=(-1, 1)) # Needed just for making y in original scale
    splitter = train_split

    # --- Data Preparation (for amplitude encoding) ---
    # Generate synthetic data with the correct number of features (2^n_sites)
    X = np.random.rand(100, 2**n_sites)  # Correct number of features for amplitude encoding
    y = np.random.rand(100)

    # Normalize data for Amplitude Encoding (sum of squares = 1)
    X_normalized = []
    for row in X:
        norm = np.linalg.norm(row)
        normalized_row = row / norm
        X_normalized.append(normalized_row)
    X_normalized = np.array(X_normalized)
    
    # Normalize y values 
    y_normalized, scaler_y = normalize(X_normalized,y)[:2]  

    # No padding needed if X already has the correct number of features
    X_train, X_val, X_test, y_train, y_val, y_test = splitter(
    X_normalized, y_normalized, test_size=0.4, random_state=402)    

    # --- QRC Workflow ---
    # Encode
    encoded_train = encoder.encode(X_train)
    encoded_test = encoder.encode(X_test)

    # Evolve
    solver = BloqadeSolver()
    evolved_train = solver.simulate(
        encoded_train, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100
    )
    evolved_test = solver.simulate(
        encoded_test, hamiltonian=hamiltonian.get_hamiltonian(), duration=time, steps=100
    )

    # Measure
    train_features = readout.measure(evolved_train)
    test_features = readout.measure(evolved_test)

    # Predict
    predictor.fit(train_features, y_train)
    predictions = predictor.predict(test_features)

    # Inverse transform to get original scale.
    predictions = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    y_test = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

    # Evaluate
    mse = mean_squared_error(y_test, predictions)
    print(f"Test MSE (Amplitude Encoding): {mse}")
    assert mse >= 0

# Cell 4: Main execution block


In [14]:
if __name__ == "__main__":
    print("Running Angle Encoding Test...")
    test_qrc_angle_encoding()
    print("\nRunning Amplitude Encoding Test...")
    test_qrc_amplitude_encoding()

Running Angle Encoding Test...
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: 'module' object is not callable
Simulation error: '

AttributeError: module 'bloqade' has no attribute 'expectation'